In [1]:
import dynamiqs as dq
from functools import partial
import matplotlib.pyplot as plt
from IPython.display import clear_output

import math
import jax
import jax.numpy as jnp
from jax import value_and_grad,grad, vmap,jit
import qcsys as qs
import jaxquantum as jqt
jax.config.update('jax_platform_name', 'gpu')
dq.set_device('gpu')

import optax
import warnings
warnings.filterwarnings("ignore")

2024-03-25 03:44:01.786950: E external/xla/xla/stream_executor/cuda/cuda_driver.cc:282] failed call to cuInit: UNKNOWN ERROR (100)


RuntimeError: Unknown backend: 'gpu' requested, but no platforms that are instances of gpu are present. Platforms present are: cpu

### let's first optimize the transmon frequency to be fluxonium 0-7 transition

In [3]:
Ec_t = 0.2
def objective(params):
    Ej_t = params[0]
    qst = qs.SingleChargeTransmon.create(
        N = 10,
        params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
        N_max_charge=30
    )
    return abs(qst.eig_systems["vals"][1] - qst.eig_systems["vals"][0] - 7.182920828753576)


optimizer = optax.adam(learning_rate=0.1) 
params = jnp.array([40.0])
opt_state = optimizer.init(params)

num_steps = 1000
for step in range(num_steps):
    val, grads = jax.value_and_grad(objective)(params)
    if step %10 == 0:
        clear_output()
        print(f"iter: {step}, val={val:.4f} grads: {grads}, params: {params}")
    if val < 1e-6:
        clear_output()
        print(f"iter: {step}, val={val:.4f} grads: {grads}, params: {params}")
        break
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

print(f'Optimized params: {params}')

iter: 162, val=0.0000 grads: [-0.10836512], params: [34.12111946]
Optimized params: [34.12111946]


# Now let's define the relative functions for our objective

In [20]:
def calculate_eig(Ns, H: jqt.Qarray):
    N_tot = math.prod(Ns)
    vals, kets = jnp.linalg.eigh(H.data)

    ketsT = kets.T

    def get_product_idx(edx):
        argmax = jnp.argmax(jnp.abs(ketsT[edx]))
        return  argmax  # product index
    edxs = jnp.arange(N_tot)
    product_indices_sorted_by_eval = vmap(get_product_idx)(edxs)
    return (vals,kets,product_indices_sorted_by_eval) # Here kets is equivalent to the S in qutip.Qobj.transform

def find_closest_dressed_index(product_index, product_indices_sorted_by_eval):
    dressed_index = jnp.argmin(jnp.abs(product_index - product_indices_sorted_by_eval))
    return dressed_index.item()

def transform_op_into_dressed_basis_jax(op_matrix: jqt.Qarray, 
                                        S: jax.Array) -> jax.Array:
    """
    Transform an operator into the dressed basis using JAX.

    Parameters:
    - op_matrix: A 2D JAX array representing the operator's matrix.
    - S: A 2D JAX array representing the dressed eigenvectors similar to the S in qutip.Qobj.transform

    Returns:
    - A 2D JAX array representing the transformed operator.
    """
    data = jnp.dot(S, jnp.dot(op_matrix.data, S.T.conj()))
    return data

def square_pulse_with_rise_fall(t,
                                args = {}):
    
    w_d = args['w_d']
    amp = args['amp']
    t_start = args.get('t_start', 0)  # Default start time is 0
    t_rise = args.get('t_rise', 0)  # Default rise time is 0 for no rise
    t_square = args.get('t_square', 0)  # Duration of constant amplitude

    def cos_modulation():
        return 2 * jnp.pi * amp * jnp.cos(w_d * 2 * jnp.pi * t)
    
    t_fall_start = t_start + t_rise + t_square  # Start of fall
    t_end = t_fall_start + t_rise  # End of the pulse

    before_pulse_start = jnp.less(t, t_start)
    during_rise_segment = jnp.logical_and(jnp.greater(t_rise, 0), jnp.logical_and(jnp.greater_equal(t, t_start), jnp.less_equal(t, t_start + t_rise)))
    constant_amplitude_segment = jnp.logical_and(jnp.greater(t, t_start + t_rise), jnp.less_equal(t, t_fall_start))
    during_fall_segment = jnp.logical_and(jnp.greater(t_rise, 0), jnp.logical_and(jnp.greater(t, t_fall_start), jnp.less_equal(t, t_end)))

    return jnp.where(before_pulse_start, 0,
                    jnp.where(during_rise_segment, jnp.sin(jnp.pi * (t - t_start) / (2 * t_rise)) ** 2 * cos_modulation(),
                            jnp.where(constant_amplitude_segment, cos_modulation(),
                                        jnp.where(during_fall_segment, jnp.sin(jnp.pi * (t_end - t) / (2 * t_rise)) ** 2 * cos_modulation(), 0))))


In [21]:
solver = dq.solver.Tsit5(
                    rtol= 1e-06,
                    atol= 1e-06,
                    safety_factor= 0.9,
                    min_factor= 0.2,
                    max_factor = 5.0,
                    max_steps = int(1e4*1000),
                )

n_lvls_fluxonium = 20
n_lvls_transmon = 4

tot_dim = n_lvls_fluxonium*n_lvls_transmon

Ej_f = 2.7
Ec_f = 0.6
El_f = 0.13

truncated_dim = 26 # will include 7,1
def truncate(data: jnp.array):
    return data[:truncated_dim,:truncated_dim]

In [22]:

def objective(params):

    Ej_t = params[0]
    pulse_length = params[1]
    amp_with_2pi = params[2]
    g_tf = 0.2

    t_rise = 30
    t_tot = t_rise + pulse_length
    tlist = jnp.linspace(0,t_tot,int(t_tot))
    

    qsf = qs.Fluxonium.create(
        n_lvls_fluxonium,
        {"Ej": Ej_f, "Ec": Ec_f, "El": El_f, "phi_ext": 0.0},
        N_pre_diag=100,
        use_linear = False
        )
    Ec_t = 0.2
    qst = qs.SingleChargeTransmon.create(
        N = n_lvls_transmon,
        params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
        N_max_charge=30
        )
    devices = [qsf, qst]
    f_indx = 0
    t_indx = 1
    Ns = [device.N for device in devices]
    tn = qs.promote(qst.ops['n'], t_indx, Ns)
    fn = qs.promote(qsf.ops["n"], f_indx, Ns)
    system = qs.System.create(devices, couplings=[
        g_tf *  fn @ tn
        ])
    system.params["g_tf"] = g_tf
    system_evals_in_product_indices, system_evecs_in_product_indices = system.calculate_eig_linear()
    system_evals_sorted, system_evecs_sorted, product_indices_sorted_by_eval = calculate_eig(Ns, system.get_H())
    driven_op = transform_op_into_dressed_basis_jax(tn, system_evecs_sorted.T)

    w_d = system_evals_in_product_indices[0,1] - system_evals_in_product_indices[0,0]
    pulse_shape_args={
        'w_d': w_d,
        'amp': amp_with_2pi/(2*jnp.pi),
        't_rise': t_rise,
        't_square': pulse_length - t_rise
    }      

    def _H(t):
        _H =  2 * jnp.pi *truncate(jnp.diag(system_evals_sorted))
        _H += truncate(driven_op) * square_pulse_with_rise_fall(t, pulse_shape_args)
        return _H 
    H =  dq.timecallable(_H)
    
    psi0_list = [truncate(dq.basis(tot_dim, find_closest_dressed_index(fl*qst.N + 0, product_indices_sorted_by_eval)))
                        for fl in range(3)]
    print(f"finished initializing")
    result = dq.sesolve(
                H = H,
                psi0 = psi0_list,
                tsave = tlist,
                solver = solver
                )
    
    f0_e = dq.expect(
                truncate(dq.basis_dm(tot_dim, find_closest_dressed_index(0*qst.N + 1, product_indices_sorted_by_eval))),
                result.states[0][-1]
                ).real
    f1_e = dq.expect(
                truncate(dq.basis_dm(tot_dim, find_closest_dressed_index(1*qst.N + 1, product_indices_sorted_by_eval))),
                result.states[1][-1]
                ).real
    f2_e = dq.expect(
                truncate(dq.basis_dm(tot_dim, find_closest_dressed_index(2*qst.N + 1, product_indices_sorted_by_eval))),
                result.states[2][-1]
                ).real
    
    return 1 - f0_e + f1_e + f2_e # we minimize the objective

In [23]:
params =  jnp.array([34.12111946,
                     250.0,
                     0.011026707187734986
                     ])

optimizer = optax.adam(learning_rate=0.1) 
opt_state = optimizer.init(params)

num_steps = 1000
for step in range(num_steps):
    val, grads = jax.value_and_grad(objective)(params)
    clear_output()
    print(f"iter: {step}, val={val:.4f} grads: {grads}, params: {params}")
    if val < 1e-4:
        break
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

print(f'Optimized params: {params}')

finished initializing
